# Meeting 2: test

Tests the pool run named in `POOL_RUN` on the test photos and pushes the results to GitHub. The Meeting 1 baselines are already tested; compare against their `test-results.json`. Weights are restored from Google Drive, and the notebook stops if any are missing.

Run only after `POOL_RUN` is chosen on validation and written down, and never change it afterwards.

In [ ]:
import subprocess, sys
from getpass import getpass
from pathlib import Path
BRANCH = "excercises/2"  # the branch holding colab/ and runs/
REPO = Path("/content/dane_syntetyczne")
if not (REPO / ".git").exists():
    t = getpass("GitHub token (hidden): ")
    subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, f"https://{t}@github.com/tmsz/dane_syntetyczne.git", str(REPO)], check=True)
if subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", BRANCH]).returncode:
    raise RuntimeError(f"Branch {BRANCH!r} is not on GitHub. On the Mac: git push -u origin {BRANCH}")
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--quiet", "origin", BRANCH], check=True)
sys.path.insert(0, str(REPO / "colab"))
import colab_tools as ct
import importlib
importlib.reload(ct)  # re-running this cell picks up pulled changes
ct.authenticate()  # Drive + GitHub prompts now, before any download or training

In [ ]:
POOL_RUN = "m2-base"  # e.g. "m2-guidance-2": the pool run kept after validation; never change after testing

CANDIDATES = ["m2-base", "m2-guidance-2", "m2-guidance-15", "m2-steps-10", "m2-prompt-simple", "m2-sdxl-turbo"]
assert POOL_RUN in CANDIDATES, f"set POOL_RUN to one of {CANDIDATES}"

ct.setup()  # pull, install, download data, check GPU
ct.restore_models(POOL_RUN)  # model.pt <- Drive
ct.test_run(POOL_RUN)
ct.check_devices([POOL_RUN])
ct.save_to_github(f"Meeting 2: test {POOL_RUN}")